# Simple expression handling in Python interface

The intent is to add an expression handler to the DISPATCH Python interface, where one can register ad hoc expressions to perform with the known 3-D variables as input.

Any variable ``v`` in the RHSs of the expressions that is not also in one of the LHSs is assumed to be evaluated as ``p.var(v)``.

With the simple expression interface, the order of expressions must be such that each LHS relies only on already defined variables. 

## Test setup

In [1]:
%pylab

Using matplotlib backend: Qt5Agg
Populating the interactive namespace from numpy and matplotlib


In [2]:
cd C:\cygwin64\home\codes\dispatch\experiments\stellar_atmospheres\python

C:\cygwin64\home\codes\dispatch\experiments\stellar_atmospheres\python


In [3]:
import dispatch
sn=dispatch.snapshot(2)
p=sn.patches[0]
p.var('d').shape

(20, 20, 20)

In [4]:
p.keys

{'letters': ['d',
  'e',
  'et',
  's',
  'px',
  'py',
  'pz',
  'bx',
  'by',
  'bz',
  'qr',
  'tt',
  'phi',
  'p1',
  'p2',
  'p3',
  'b1',
  'b2',
  'b3'],
 'numbers': [0, 4, -1, 4, 1, 2, 3, 5, 6, 7, -1, -1, -1, 1, 2, 3, 5, 6, 7],
 'expressions': ['ux', 'uy', 'uz', 'u1', 'u2', 'u3', 'ee', 'E'],
 'aux': dict_keys(['rkap', 'src', 'tt', 'qr'])}

## Expression class

In [5]:
import numpy as np

class Expressions(object):
    def __init__(self,patch=None,verbose=0):
        """ Initialize Exxpressions class """
        self.verbose=verbose
        self.expressions=[]
        self.ops=' +-*/()'
        self.known=[]
        self.pvars=[]
        
        """ If a patch is given, collect all keys it accepts, except integers """
        if patch:
            for k,v in patch.keys.items():
                if k != 'numbers':
                    self.pvars.extend(v)
            if self.verbose:
                print('patch variables:',self.pvars)
        if self.verbose>1:
            print('EXPRESSIONS:')

    def split(self,rhs):
        """ Split a RHS into words, based on operators in self.ops """
        words=[]
        def add(w):
            if w != '' and w not in words:
                words.append(w) 
        word=''
        for c in rhs:
            if c in self.ops:
                add(word)
                word=''
            else:
                word+=c
        add(word)
        return words

    def register(self,expression):
        """ Register expressions, checking each RHS for words that are either
            known patch variables, previsouly defined, or scalars
        """
        if self.verbose>1:
            print(' ',expression)
        lhs,rhs=expression.split('=')
        self.expressions.append((lhs,rhs))
        for word in self.split(rhs):
            if word in self.known or word in self.pvars:
                continue
            else:
                if word in globals():
                    print('WARNING: {} in globals()'.format(word))
                try:
                    eval(word)
                    self.known.append(word)
                except:
                    print('ERROR: could not evaluate',word)
                    return False
        self.known.append(lhs)
        return True
        
    def evaluate(self):
        """ After all expressions registered, evaluate """
        if self.verbose:
            print('EVALUATE:')
        ll=locals()
        results={}
        for lhs,rhs in self.expressions:
            for word in self.split(rhs):
                if not (word in self.known or word in ll):
                    if word in self.pvars:
                        cmd="p.var('{}')".format(word)
                        try:
                                print('  {}={}'.format(word,cmd))
                                ll[word]=eval(cmd)
                        except:
                            print('ERROR: {}'.format(cmd))
                            return False
                    else:
                        print('ERROR: {} not known'.format(word))
                    
            """ Now the whole RHS should be possible to evaluate """
            if self.verbose:
                print('  {}={}'.format(lhs,rhs))
            ll[lhs]=eval(rhs)
            results[lhs]=ll[lhs]
            self.known.append(lhs)
        return results

### Test

First with only scalars:

In [6]:
expressions=Expressions()
expressions.register('c=np.sqrt(4)')
expressions.register('b=c+1.5')
expressions.register('a=b+c')
expressions.evaluate()

{'c': 2.0, 'b': 3.5, 'a': 5.5}

With external scalar:

In [7]:
d=10
expressions=Expressions(verbose=2)
expressions.register('c=np.sqrt(4)')
expressions.register('b=c+1.5')
expressions.register('a=b+d')
expressions.evaluate()

EXPRESSIONS:
  c=np.sqrt(4)
  b=c+1.5
  a=b+d
EVALUATE:
  c=np.sqrt(4)
  b=c+1.5
  a=b+d


{'c': 2.0, 'b': 3.5, 'a': 13.5}

Global variables are recognized, so delete any that should not be there:

In [8]:
del d

Now with ``d`` from a patch:

In [9]:
expressions=Expressions(p,verbose=1)
expressions.register('c=np.sqrt(4)')
expressions.register('b=c+1.5')
expressions.register('a=b+d')
expressions.evaluate()

patch variables: ['d', 'e', 'et', 's', 'px', 'py', 'pz', 'bx', 'by', 'bz', 'qr', 'tt', 'phi', 'p1', 'p2', 'p3', 'b1', 'b2', 'b3', 'ux', 'uy', 'uz', 'u1', 'u2', 'u3', 'ee', 'E', 'rkap', 'src', 'tt', 'qr']
EVALUATE:
  c=np.sqrt(4)
  b=c+1.5
  d=p.var('d')
  a=b+d


{'c': 2.0,
 'b': 3.5,
 'a': array([[[ 3.6098187,  3.6736176,  3.7662528, ..., 10.481551 ,
          11.525787 , 12.636501 ],
         [ 3.6093504,  3.6726635,  3.7647066, ..., 10.578565 ,
          11.609745 , 12.693161 ],
         [ 3.6090527,  3.6720753,  3.763824 , ..., 10.664262 ,
          11.674124 , 12.727867 ],
         ...,
         [ 3.5714664,  3.6157455,  3.6780894, ...,  9.881198 ,
          10.643985 , 11.557012 ],
         [ 3.5666451,  3.6086888,  3.667572 , ...,  9.795718 ,
          10.587722 , 11.542337 ],
         [ 3.5620446,  3.6019034,  3.6580288, ...,  9.767666 ,
          10.604405 , 11.60629  ]],
 
        [[ 3.6170611,  3.6846864,  3.7846773, ..., 10.766865 ,
          11.780383 , 12.777465 ],
         [ 3.6171732,  3.684478 ,  3.7843893, ..., 10.856383 ,
          11.859919 , 12.821154 ],
         [ 3.6174045,  3.6845362,  3.7845387, ..., 10.929323 ,
          11.910524 , 12.833712 ],
         ...,
         [ 3.5795927,  3.6276758,  3.6967642, ...,  9.987534

The expression handling should be modified into a library module, imported into ``_dispatch.py``